In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler, Imputer
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [4]:
spark = SparkSession.builder \
    .appName("ML_Feature_Engineering_Test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 09:37:34 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/24 09:37:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 09:37:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/24 09:37:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("age", DoubleType(), True),
    StructField("income", DoubleType(), True),
    StructField("city", StringType(), True)
])
data = [
    (1, 25.0, 50000.0, "New York"),
    (2, 45.0, None,    "San Francisco"), # Missing income
    (3, None, 120000.0, "Los Angeles"),  # Missing age
    (4, 34.0, 80000.0, "New York"),
    (5, 50.0, 150000.0, "Chicago"),
    (6, 29.0, 60000.0, "San Francisco"),
    (7, None, None,    "Los Angeles"),   # Missing both!
    (8, 42.0, 95000.0, "Chicago"),
    (9, 38.0, 105000.0, "New York"),
    (10, 22.0, 45000.0, "Chicago")
]
df = spark.createDataFrame(data, schema)
display(df.show())

+---+----+--------+-------------+
| id| age|  income|         city|
+---+----+--------+-------------+
|  1|25.0| 50000.0|     New York|
|  2|45.0|    NULL|San Francisco|
|  3|NULL|120000.0|  Los Angeles|
|  4|34.0| 80000.0|     New York|
|  5|50.0|150000.0|      Chicago|
|  6|29.0| 60000.0|San Francisco|
|  7|NULL|    NULL|  Los Angeles|
|  8|42.0| 95000.0|      Chicago|
|  9|38.0|105000.0|     New York|
| 10|22.0| 45000.0|      Chicago|
+---+----+--------+-------------+



None

In [ ]:
"""
In PySpark, VectorAssembler is a feature transformer that combines a given list of multiple numeric, boolean, or vector columns into a single, unified vector column. It is a mandatory preprocessing step because almost all Machine Learning algorithms in spark.ml require the input features to be stored in a single vector per row.

Core Use CasesPreparing Data for ML Models: 
Grouping independent variables (e.g., age, income, years_experience) into a single features column to pass into models like LinearRegression or RandomForestClassifier.
Combining Raw & Engineered Features: Merging base numerical variables with features generated from other transformers (such as OneHotEncoder for categories, or Word2Vec for text).
Machine Learning Pipelines: Chaining VectorAssembler as a stage in a Spark Pipeline so that data processing steps are automatically applied to both training and test datasets.
"""

In [8]:
# applying VectorAssembler which combines the features into a single vector column
assembler = VectorAssembler(
    inputCols=["age", 'income'],
    outputCol="features",
    handleInvalid="skip"
)

assembled_df = assembler.transform(df)
assembled_df.show()

+---+----+--------+-------------+---------------+
| id| age|  income|         city|       features|
+---+----+--------+-------------+---------------+
|  1|25.0| 50000.0|     New York| [25.0,50000.0]|
|  4|34.0| 80000.0|     New York| [34.0,80000.0]|
|  5|50.0|150000.0|      Chicago|[50.0,150000.0]|
|  6|29.0| 60000.0|San Francisco| [29.0,60000.0]|
|  8|42.0| 95000.0|      Chicago| [42.0,95000.0]|
|  9|38.0|105000.0|     New York|[38.0,105000.0]|
| 10|22.0| 45000.0|      Chicago| [22.0,45000.0]|
+---+----+--------+-------------+---------------+



In [11]:
indexer = StringIndexer(
    inputCol="city", 
    outputCol="city_indexed",
    handleInvalid="keep"
)

indexed_df = indexer.fit(assembled_df).transform(assembled_df)
display(indexed_df.show())

encoder = OneHotEncoder(
    inputCol="city_indexed", 
    outputCol="city_vec", 
    dropLast=True
)
encoded_df = encoder.fit(indexed_df).transform(indexed_df)
display(encoded_df.show())

+---+----+--------+-------------+---------------+------------+
| id| age|  income|         city|       features|city_indexed|
+---+----+--------+-------------+---------------+------------+
|  1|25.0| 50000.0|     New York| [25.0,50000.0]|         1.0|
|  4|34.0| 80000.0|     New York| [34.0,80000.0]|         1.0|
|  5|50.0|150000.0|      Chicago|[50.0,150000.0]|         0.0|
|  6|29.0| 60000.0|San Francisco| [29.0,60000.0]|         2.0|
|  8|42.0| 95000.0|      Chicago| [42.0,95000.0]|         0.0|
|  9|38.0|105000.0|     New York|[38.0,105000.0]|         1.0|
| 10|22.0| 45000.0|      Chicago| [22.0,45000.0]|         0.0|
+---+----+--------+-------------+---------------+------------+



None

+---+----+--------+-------------+---------------+------------+-------------+
| id| age|  income|         city|       features|city_indexed|     city_vec|
+---+----+--------+-------------+---------------+------------+-------------+
|  1|25.0| 50000.0|     New York| [25.0,50000.0]|         1.0|(3,[1],[1.0])|
|  4|34.0| 80000.0|     New York| [34.0,80000.0]|         1.0|(3,[1],[1.0])|
|  5|50.0|150000.0|      Chicago|[50.0,150000.0]|         0.0|(3,[0],[1.0])|
|  6|29.0| 60000.0|San Francisco| [29.0,60000.0]|         2.0|(3,[2],[1.0])|
|  8|42.0| 95000.0|      Chicago| [42.0,95000.0]|         0.0|(3,[0],[1.0])|
|  9|38.0|105000.0|     New York|[38.0,105000.0]|         1.0|(3,[1],[1.0])|
| 10|22.0| 45000.0|      Chicago| [22.0,45000.0]|         0.0|(3,[0],[1.0])|
+---+----+--------+-------------+---------------+------------+-------------+



None

In [16]:
# -- MIN-MAX SCALING --
min_max_scaler = MinMaxScaler(
    inputCol="features", 
    outputCol="scaled_minmax_features",
    min=0.0, 
    max=1.0
)
scaled_df_mm = min_max_scaler.fit(assembled_df).transform(assembled_df)

# -- STANDARD SCALING --
standard_scaler = StandardScaler(
    inputCol="features", 
    outputCol="scaled_std_features",
    withMean=True, 
    withStd=True
)
scaled_df_std = standard_scaler.fit(assembled_df).transform(assembled_df)
display(scaled_df_mm.show())
display(scaled_df_std.show())

+---+----+--------+-------------+---------------+----------------------+
| id| age|  income|         city|       features|scaled_minmax_features|
+---+----+--------+-------------+---------------+----------------------+
|  1|25.0| 50000.0|     New York| [25.0,50000.0]|  [0.10714285714285...|
|  4|34.0| 80000.0|     New York| [34.0,80000.0]|  [0.42857142857142...|
|  5|50.0|150000.0|      Chicago|[50.0,150000.0]|  [1.0,0.9999999999...|
|  6|29.0| 60000.0|San Francisco| [29.0,60000.0]|  [0.25,0.142857142...|
|  8|42.0| 95000.0|      Chicago| [42.0,95000.0]|  [0.71428571428571...|
|  9|38.0|105000.0|     New York|[38.0,105000.0]|  [0.57142857142857...|
| 10|22.0| 45000.0|      Chicago| [22.0,45000.0]|             [0.0,0.0]|
+---+----+--------+-------------+---------------+----------------------+



None

+---+----+--------+-------------+---------------+--------------------+
| id| age|  income|         city|       features| scaled_std_features|
+---+----+--------+-------------+---------------+--------------------+
|  1|25.0| 50000.0|     New York| [25.0,50000.0]|[-0.9400565608442...|
|  4|34.0| 80000.0|     New York| [34.0,80000.0]|[-0.0289248172567...|
|  5|50.0|150000.0|      Chicago|[50.0,150000.0]|[1.59086494912100...|
|  6|29.0| 60000.0|San Francisco| [29.0,60000.0]|[-0.5351091192497...|
|  8|42.0| 95000.0|      Chicago| [42.0,95000.0]|[0.78097006593212...|
|  9|38.0|105000.0|     New York|[38.0,105000.0]|[0.37602262433769...|
| 10|22.0| 45000.0|      Chicago| [22.0,45000.0]|[-1.2437671420400...|
+---+----+--------+-------------+---------------+--------------------+



None

In [18]:
imputer = Imputer(
    inputCols=["age", "income"], 
    outputCols=["age_imputed", "income_imputed"]
).setStrategy("median")

# Fit calculates the median across the distributed dataset
# Transform fills the nulls with the calculated medians
imputed_df = imputer.fit(df).transform(df)
imputed_df.show()

+---+----+--------+-------------+-----------+--------------+
| id| age|  income|         city|age_imputed|income_imputed|
+---+----+--------+-------------+-----------+--------------+
|  1|25.0| 50000.0|     New York|       25.0|       50000.0|
|  2|45.0|    NULL|San Francisco|       45.0|       80000.0|
|  3|NULL|120000.0|  Los Angeles|       34.0|      120000.0|
|  4|34.0| 80000.0|     New York|       34.0|       80000.0|
|  5|50.0|150000.0|      Chicago|       50.0|      150000.0|
|  6|29.0| 60000.0|San Francisco|       29.0|       60000.0|
|  7|NULL|    NULL|  Los Angeles|       34.0|       80000.0|
|  8|42.0| 95000.0|      Chicago|       42.0|       95000.0|
|  9|38.0|105000.0|     New York|       38.0|      105000.0|
| 10|22.0| 45000.0|      Chicago|       22.0|       45000.0|
+---+----+--------+-------------+-----------+--------------+

